In [112]:
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError
import time
from dotenv import load_dotenv
import os
from urllib.parse import quote_plus
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [113]:
# LOAD ENVIRONEMNET VARIABLES
load_dotenv()

# DATABASE CONFIGURATION
MYSQL_USER = "root"
MYSQL_PASSWORD = quote_plus(os.getenv("password"))
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DATABASE = "etl_capstone"

# SQLAlchemy Connection URL
DATABASE_URL = (
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

# Create Engine
engine = create_engine(DATABASE_URL)

In [114]:
df = pd.read_sql("select * from enriched_posts", con=engine)

df.head()

,id,userId,title,body,title_word_count,body_word_count,total_content_length,score_category,engagement_score
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...,9,23,232,Medium,41
1,2,1,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...,3,31,218,Medium,37
2,3,1,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...,9,26,223,Medium,44
3,4,1,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...,4,28,210,Medium,36
4,5,1,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...,3,23,165,Medium,29


In [115]:
print("Shape:", df.shape)

df.info()

df.isnull().sum()



Shape: (100, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    100 non-null    int64 
 1   userId                100 non-null    int64 
 2   title                 100 non-null    object
 3   body                  100 non-null    object
 4   title_word_count      100 non-null    int64 
 5   body_word_count       100 non-null    int64 
 6   total_content_length  100 non-null    int64 
 7   score_category        100 non-null    object
 8   engagement_score      100 non-null    int64 
dtypes: int64(6), object(3)
memory usage: 7.2+ KB


id                      0
userId                  0
title                   0
body                    0
title_word_count        0
body_word_count         0
total_content_length    0
score_category          0
engagement_score        0
dtype: int64

The dataset was inspected to identify:
- Number of rows and columns
- Data types (numerical and categorical)
- Missing values
- Potential low-value columns

In [116]:
df.columns



Index(['id', 'userId', 'title', 'body', 'title_word_count', 'body_word_count',
       'total_content_length', 'score_category', 'engagement_score'],
      dtype='object')

In [117]:
# check unique values
for col in df.columns:
    print(f"{col}")
    print(df[col].nunique())

id
100
userId
10
title
100
body
100
title_word_count
7
body_word_count
17
total_content_length
69
score_category
2
engagement_score
24


In [118]:
# Dropping identifier columns

drop_cols = ['id', 'userId']

df = df.drop(columns=drop_cols)

df.head()

,title,body,title_word_count,body_word_count,total_content_length,score_category,engagement_score
0,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...,9,23,232,Medium,41
1,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...,3,31,218,Medium,37
2,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...,9,26,223,Medium,44
3,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...,4,28,210,Medium,36
4,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...,3,23,165,Medium,29


Decision 1: Dropping Low-Value Columns

1. id
   - Unique identifier for each record.
   - Does not provide predictive information.

2. userId
   - Represents user identification.
   - Acts as a reference key rather than a meaningful feature.

These columns were removed to avoid introducing noise into the model.

In [119]:
# separate features and target variable
target_column = 'engagement_score'

X = df.drop(columns=[target_column])
y = df[target_column]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (100, 6)
Target Shape: (100,)


Decision 2: Target Variable Selection

The column 'engagement_score' was selected as the target variable because it represents the outcome that the model will predict.

In [120]:
# identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)

Numerical Columns:
['title_word_count', 'body_word_count', 'total_content_length']

Categorical Columns:
['title', 'body', 'score_category']


In [121]:
print(X.isnull().sum()) # checking missing values in features

title                   0
body                    0
title_word_count        0
body_word_count         0
total_content_length    0
score_category          0
dtype: int64


In [122]:
# define imputation strategies

# Numerical columns -> Median
num_imputer = SimpleImputer(strategy='median')

# Categorical columns -> Most Frequent
cat_imputer = SimpleImputer(strategy='most_frequent')

Decision 4: Missing Value Imputation

Two different imputation strategies were used:

1. Median Imputation
   - Applied to numerical columns.
   - Less sensitive to outliers than mean imputation.

2. Most Frequent Imputation
   - Applied to categorical columns.
   - Replaces missing values with the most common category.

In [123]:
# encoding strategies
categorical_cols = ['score_category']

encoder = OneHotEncoder(handle_unknown='ignore')

Decision 5: Encoding Categorical Variables

The 'score_category' column contains nominal categories and was encoded using One-Hot Encoding.

The text columns ('title' and 'body') were excluded from One-Hot Encoding because they contain many unique values, which would create an extremely large number of features.

In [124]:
# remove text columns for scaling
X = X.drop(columns=['title', 'body'])

print(X.head(20))

    title_word_count  body_word_count  total_content_length score_category
0                  9               23                   232         Medium
1                  3               31                   218         Medium
2                  9               26                   223         Medium
3                  4               28                   210         Medium
4                  3               23                   165         Medium
5                  6               26                   228         Medium
6                  3               27                   165         Medium
7                  4               26                   200         Medium
8                  7               23                   186         Medium
9                  5               21                   156         Medium
10                 6               25                   201         Medium
11                 6               26                   189         Medium
12                 9     

Decision 6: Removing Text Features

The columns 'title' and 'body' contain free-form text.

Since this assignment focuses on standard preprocessing techniques and not NLP, these columns were removed to avoid generating thousands of sparse encoded features.

In [125]:
# reidintify columns type
numerical_cols = [
    'title_word_count',
    'body_word_count',
    'total_content_length'
]

categorical_cols = [
    'score_category'
]

In [133]:
# define standard scaler

scaler = StandardScaler()


Decision 7: Feature Scaling

StandardScaler was applied to numerical features.

Benefits:
- Mean becomes 0.
- Standard deviation becomes 1.
- Improves model performance and convergence.

In [127]:
# preprocessing pipelines

numeric_transformer = Pipeline(
    steps=[
        ('imputer', num_imputer),
        ('scaler', scaler)
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', cat_imputer),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [ ]:
# train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 80
Testing Samples: 20


Decision 8: Train-Test Split

The dataset was divided into:
- 80% Training Data
- 20% Testing Data

This ensures that the model is evaluated on unseen data while retaining sufficient data for training.

In [129]:
# apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [130]:
# convert to dataframe
encoded_cols = preprocessor.named_transformers_['cat'] \
                           .named_steps['encoder'] \
                           .get_feature_names_out(categorical_cols)

all_columns = numerical_cols + list(encoded_cols)

X_train_clean = pd.DataFrame(
    X_train_processed,
    columns=all_columns
)

X_test_clean = pd.DataFrame(
    X_test_processed,
    columns=all_columns
)

In [131]:
print("X_train_clean Shape:", X_train_clean.shape)
print("X_test_clean Shape:", X_test_clean.shape)

display(X_train_clean.head())
display(X_test_clean.head())

print("\ny_train")
display(y_train.head())

print("\ny_test")
display(y_test.head())

X_train_clean Shape: (80, 5)
X_test_clean Shape: (20, 5)


,title_word_count,body_word_count,total_content_length,score_category_Medium,score_category_Short
0,-0.629428,-0.513102,-0.426254,1.0,0.0
1,1.536346,-0.513102,-0.246463,1.0,0.0
2,0.994902,0.811032,0.802316,1.0,0.0
3,1.536346,1.340685,2.030886,1.0,0.0
4,-1.712315,-1.307581,-1.475033,0.0,1.0


,title_word_count,body_word_count,total_content_length,score_category_Medium,score_category_Short
0,1.536346,-1.042755,0.292909,1.0,0.0
1,0.453459,-1.837235,-0.815800,0.0,1.0
2,0.453459,-0.513102,-0.875731,1.0,0.0
3,-0.087985,-2.102061,-1.594893,0.0,1.0
4,0.994902,-1.572408,-0.725905,0.0,1.0



y_train


55    32
88    40
26    43
42    47
69    25
Name: engagement_score, dtype: int64


y_test


83    38
53    31
70    36
45    28
44    34
Name: engagement_score, dtype: int64

Preprocessing Summary

1. Removed low-value columns:
   - id
   - userId

2. Missing values handled using:
   - Median Imputation (Numerical)
   - Most Frequent Imputation (Categorical)

3. Encoded categorical feature:
   - score_category using One-Hot Encoding

4. Removed high-cardinality text fields:
   - title
   - body

5. Scaled numerical features:
   - title_word_count
   - body_word_count
   - total_content_length

6. Performed train-test split:
   - 80% Training
   - 20% Testing

Final Outputs:
✓ X_train_clean
✓ X_test_clean
✓ y_train
✓ y_test